In [1]:
# ==========================================
# 1) Imports
# ==========================================
import pandas as pd
import numpy as np
from google.colab import files

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ==========================================
# 2) Upload raw DVF file
# ==========================================
uploaded = files.upload()
raw_filename = list(uploaded.keys())[0]
print("Uploaded file:", raw_filename)

# ==========================================
# 3) Column names from DVF sample
# ==========================================
columns = [
    "id_mutation",
    "date_mutation",
    "numero_disposition",
    "nature_mutation",
    "valeur_fonciere",
    "adresse_numero",
    "adresse_suffixe",
    "adresse_nom_voie",
    "adresse_code_voie",
    "code_postal",
    "code_commune",
    "nom_commune",
    "code_departement",
    "ancien_code_commune",
    "ancien_nom_commune",
    "id_parcelle",
    "ancien_id_parcelle",
    "numero_volume",
    "lot1_numero",
    "lot1_surface_carrez",
    "lot2_numero",
    "lot2_surface_carrez",
    "lot3_numero",
    "lot3_surface_carrez",
    "lot4_numero",
    "lot4_surface_carrez",
    "lot5_numero",
    "lot5_surface_carrez",
    "nombre_lots",
    "code_type_local",
    "type_local",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "code_nature_culture",
    "nature_culture",
    "code_nature_culture_speciale",
    "nature_culture_speciale",
    "surface_terrain",
    "longitude",
    "latitude",
    "section_prefixe",
]

print("Number of expected columns:", len(columns))

# ==========================================
# 4) Read the raw file
# ==========================================
# Your sample looks tab-separated.
# If this fails, we try a fallback with automatic separator detection.
try:
    df = pd.read_csv(
        raw_filename,
        sep="\t",
        header=None,
        names=columns,
        na_values=["None", "nan", "NaN", ""],
        engine="python"
    )
    print("Loaded with TAB separator.")
except Exception as e:
    print("TAB read failed, trying auto-detection...")
    df = pd.read_csv(
        raw_filename,
        sep=None,
        header=None,
        names=columns,
        na_values=["None", "nan", "NaN", ""],
        engine="python"
    )
    print("Loaded with automatic separator detection.")

print("\nRaw shape:", df.shape)
print("\nFirst 5 raw rows:")
display(df.head())

print("\nNon-null counts before cleaning:")
display(df.notna().sum())

# ==========================================
# 5) Basic string cleanup
# ==========================================
for col in ["nature_mutation", "type_local", "code_departement"]:
    df[col] = df[col].astype(str).str.strip()

# Replace fake string nulls after astype(str)
df = df.replace({"None": np.nan, "nan": np.nan, "NaN": np.nan, "": np.nan})

# ==========================================
# 6) Convert numeric/date columns
# ==========================================
numeric_cols = [
    "valeur_fonciere",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "code_postal",
    "surface_terrain",
    "longitude",
    "latitude",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")

print("\nUnique values in nature_mutation:")
print(df["nature_mutation"].dropna().unique()[:20])

print("\nUnique values in type_local:")
print(df["type_local"].dropna().unique()[:20])

# ==========================================
# 7) Keep only residential sales
# ==========================================
clean = df.copy()

# Only sales
clean = clean[clean["nature_mutation"] == "Vente"]

# Only residential property rows
clean = clean[clean["type_local"].isin(["Appartement", "Maison"])]

# Keep only rows with the minimal CESAR columns available
required_cols = [
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "code_departement",
    "type_local",
    "valeur_fonciere",
]
clean = clean.dropna(subset=required_cols)

# Remove impossible values
clean = clean[clean["surface_reelle_bati"] > 0]
clean = clean[clean["nombre_pieces_principales"] > 0]
clean = clean[clean["valeur_fonciere"] > 0]

# Optional soft outlier filtering for a first baseline
clean = clean[clean["surface_reelle_bati"] <= 500]
clean = clean[clean["valeur_fonciere"] <= 5_000_000]

print("\nShape after cleaning:", clean.shape)
print("\nSample after cleaning:")
display(clean.head(10))

# ==========================================
# 8) Create minimal CESAR training CSV
# ==========================================
cesar_minimal = clean[
    [
        "surface_reelle_bati",
        "nombre_pieces_principales",
        "code_departement",
        "type_local",
        "valeur_fonciere",
    ]
].copy()

# Reset index for cleanliness
cesar_minimal = cesar_minimal.reset_index(drop=True)

print("\nFinal CESAR dataset shape:", cesar_minimal.shape)
display(cesar_minimal.head(10))

print("\nMissing values in final file:")
print(cesar_minimal.isna().sum())

print("\nValue counts for type_local:")
print(cesar_minimal["type_local"].value_counts())

# ==========================================
# 9) Save cleaned file
# ==========================================
output_file = "dvf_clean_minimal.csv"
cesar_minimal.to_csv(output_file, sep=";", index=False)

print(f"\nSaved cleaned training file: {output_file}")

# Download result
files.download(output_file)

Saving dvf_75115_000CM.csv to dvf_75115_000CM.csv
Uploaded file: dvf_75115_000CM.csv
Number of expected columns: 41
TAB read failed, trying auto-detection...
Loaded with automatic separator detection.

Raw shape: (51, 41)

First 5 raw rows:


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe
0,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe
1,2025-500599,2025-03-28,1,Vente,620100.0,22,NaN,RUE ANDRE GIDE,0324,75015,75115,Paris 15e Arrondissement,75,NaN,NaN,75115000CM0072,NaN,NaN,14,67.23,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,Appartement,67.0,3.0,NaN,NaN,NaN,NaN,NaN,2.312621,48.837053,000CM
2,2025-500599,2025-03-28,1,Vente,620100.0,22,NaN,RUE ANDRE GIDE,0324,75015,75115,Paris 15e Arrondissement,75,NaN,NaN,75115000CM0072,NaN,NaN,14,67.23,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,3,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.312621,48.837053,000CM
3,2023-1355458,2023-10-11,1,Vente,490000.0,22,NaN,RUE ANDRE GIDE,0324,75015,75115,Paris 15e Arrondissement,75,NaN,NaN,75115000CM0072,NaN,NaN,30,48.35,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,Appartement,48.0,2.0,NaN,NaN,NaN,NaN,NaN,2.312621,48.837053,000CM
4,2023-1355458,2023-10-11,1,Vente,490000.0,22,NaN,RUE ANDRE GIDE,0324,75015,75115,Paris 15e Arrondissement,75,NaN,NaN,75115000CM0072,NaN,NaN,30,48.35,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,3,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.312621,48.837053,000CM



Non-null counts before cleaning:


,0
id_mutation,51
date_mutation,51
numero_disposition,51
nature_mutation,51
valeur_fonciere,51
adresse_numero,51
adresse_suffixe,1
adresse_nom_voie,51
adresse_code_voie,51
code_postal,51



Unique values in nature_mutation:
['nature_mutation' 'Vente']

Unique values in type_local:
['type_local' 'Appartement' 'Dépendance']

Shape after cleaning: (18, 41)

Sample after cleaning:


...


Final CESAR dataset shape: (18, 5)


,surface_reelle_bati,nombre_pieces_principales,code_departement,type_local,valeur_fonciere
0,67.0,3.0,75,Appartement,620100.0
1,48.0,2.0,75,Appartement,490000.0
2,66.0,3.0,75,Appartement,712200.0
3,45.0,2.0,75,Appartement,550000.0
4,120.0,5.0,75,Appartement,660000.0
5,104.0,4.0,75,Appartement,1377500.0
6,29.0,1.0,75,Appartement,305825.0
7,28.0,2.0,75,Appartement,440000.0
8,95.0,4.0,75,Appartement,914180.0
9,71.0,3.0,75,Appartement,890000.0



Missing values in final file:
surface_reelle_bati          0
nombre_pieces_principales    0
code_departement             0
type_local                   0
valeur_fonciere              0
dtype: int64

Value counts for type_local:
type_local
Appartement    18
Name: count, dtype: int64

Saved cleaned training file: dvf_clean_minimal.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
df = df[df["id_mutation"] != "id_mutation"].copy()